# Setup: Config & Output Tables


## Imports

In [0]:
import pyspark.sql.functions as F

## Create the config (control) table
`hash_config` drives which tables get hashed, which columns form the primary key / composite key, and which hash algorithm to use.

In [0]:
config_table = "moh_raz.raz_casemix_raw.hash_config"

In [0]:
spark.sql(f"""
DROP TABLE IF EXISTS {config_table}
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {config_table} (
    catalog         STRING,
    schema          STRING,
    table           STRING,
    primary_key     STRING,
    composite_keys  STRING,
    hash_algorithm  STRING,
    is_active       INT
)
USING DELTA
""")

### Seed sample config rows
Uses `MERGE` so re-running this cell is idempotent (no duplicate rows).

In [0]:
sample_config = [
    # ("iics", "casemix", "healthcare_ingestion_test", "CASE_SURR_ID",
    #  "CASE_SURR_ID,PATIENT_SURR_ID,CASE_TYPE,HEALTHCARE_FACILITY,PATIENT_AC_NO,DIAG_CODE,DIAG_TYPE,MDC,DRG",
    #  "md5", 0),
    # ("iics", "casemix", "healthcare_merge_test", "CASE_SURR_ID",
    #  "CASE_SURR_ID,PATIENT_SURR_ID,CASE_TYPE,HEALTHCARE_FACILITY,PATIENT_AC_NO,DIAG_CODE,DIAG_TYPE,MDC,DRG",
    #  "sha256", 0),
    # ("iics", "casemix", "sample_hash_data_test", "customer_id",
    #  "customer_id,name,dob,status",
    #  "md5", 0),
    
    # ("moh_raz", "raz_casemix_raw", "T_OD_cs_drg_rw_alos_icd10_test", "drg",
    #  "drg,mdc,drg_desc,drg_type,dh_created_date,dh_updated_date",
    #  "md5", 1)
    ("iics", "casemix", "t_od_cs_hospitall_1_test", "HEALTHCARE_FACILITY",
     "HEALTHCARE_FACILITY,HOSPITAL_CODE,HOSPITAL_DESC,HOSPITAL_TYPE,HOSPITAL_CENTER,START_YEAR,END_YEAR,DH_CREATED_DATE,DH_UPDATED_DATE",
     "md5", 1),
    
]

sample_config_df = spark.createDataFrame(
    sample_config,
    schema="catalog STRING, schema STRING, table STRING, primary_key STRING,composite_keys STRING, hash_algorithm STRING, is_active INT"
)

In [0]:
sample_config_df.write.format("delta").mode("overwrite").saveAsTable(f"{config_table}")

## Create the output (hashed values) table
One row is written per source row, per configured table. Columns ending in `___c` (e.g. `loaded_at___c`) are metadata columns and are always excluded from the columns that get hashed.

In [0]:
tgt_catalog = "moh_raz" 
tgt_schema  = "raz_casemix_raw"
tgt_table   = "hashed_values"

In [0]:
spark.sql(f"""
DROP TABLE IF EXISTS {tgt_catalog}.{tgt_schema}.{tgt_table}
""")

In [0]:
output_table = f"{tgt_catalog}.{tgt_schema}.{tgt_table}"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {output_table} (
    catalog                     STRING,
    schema                      STRING,
    table                       STRING,
    primary_key                 STRING,
    primary_key_hash_value      STRING,
    composite_keys              STRING,
    composite_keys_hash_value   STRING,
    hash_value                  STRING,
    hash_algo                   STRING,
    loaded_at___c               TIMESTAMP
)
USING DELTA
""")

print(f"Output table ready: {output_table}")

In [0]:
%sql
SELECT * FROM moh_raz.raz_casemix_raw.hash_config